# Task h - Part 2
Select one target for decarbonisation (i.e., one CO2 allowance limit). What is the (global/system-wide) CO2 price required to achieve that decarbonisation level? Search for information on the existing CO2 tax in your countries (if any) and discuss your results. Is the model in agreement with the existing CO2 tax (either national CO2 tax and/or the European CO2 price coming from the ETS)? Why or why not?

In [27]:
import pypsa
import pandas as pd
import numpy as np
pd.options.mode.string_storage = "python"

# 1. Load datasets 
data_solar = pd.read_csv('data/pv_optimal.csv', sep=';', index_col=0, parse_dates=True)
data_wind = pd.read_csv('data/onshore_wind_1979-2017.csv', sep=';', index_col=0, parse_dates=True)
data_el = pd.read_csv('data/electricity_demand.csv', sep=';', index_col=0, parse_dates=True)

# 2. Setup Costs 
year_cost = 2030
url = f"https://raw.githubusercontent.com/PyPSA/technology-data/v0.11.0/outputs/costs_{year_cost}.csv"
costs = pd.read_csv(url, index_col=[0, 1])

# 3. Process Costs
costs.loc[costs.unit.str.contains("/kW"), "value"] *= 1e3
defaults = {"FOM": 0, "VOM": 0, "efficiency": 1, "fuel": 0, "investment": 0, "lifetime": 25, "discount rate": 0.07}
costs = costs.value.unstack().fillna(defaults)
costs.at["CCGT", "fuel"] = costs.at["gas", "fuel"]

def annuity(r, n):
    return r / (1.0 - 1.0 / (1.0 + r) ** n)

costs["marginal_cost"] = costs["VOM"] + costs["fuel"] / costs["efficiency"]
ann = costs.apply(lambda x: annuity(x["discount rate"], x["lifetime"]), axis=1)
costs["capital_cost"] = (ann + costs["FOM"] / 100) * costs["investment"]

print("Costs and data loaded successfully.")

Costs and data loaded successfully.


In [28]:
n = pypsa.Network()

# Snapshot for Spain 2011
snapshots = pd.date_range("2011-01-01 00:00", "2011-12-31 23:00", freq="h")
n.set_snapshots(snapshots)

# CO2 emission factors (Source: Problem 9.1)
n.add("Carrier", "coal", co2_emissions=0.336, color="indianred")
n.add("Carrier", "CCGT", co2_emissions=0.198, color="yellow-green")
n.add("Carrier", "onwind", co2_emissions=0, color="dodgerblue")
n.add("Carrier", "solar", co2_emissions=0, color="gold")

n.add("Bus", "Spain electricity")

In [29]:
# Add Generators
n.add("Generator", "coal", bus="Spain electricity", carrier="coal",
      p_nom_extendable=True, p_nom_max=11700,
      capital_cost=costs.at["coal", "capital_cost"], 
      marginal_cost=costs.at["coal", "marginal_cost"])

n.add("Generator", "CCGT", bus="Spain electricity", carrier="CCGT",
      p_nom_extendable=True, p_nom_max=25300,
      capital_cost=costs.at["CCGT", "capital_cost"],
      marginal_cost=costs.at["CCGT", "marginal_cost"])

# Add Renewables 
n.add("Generator", "onwind", bus="Spain electricity", carrier="onwind",
      p_nom_extendable=True, capital_cost=costs.at["onwind", "capital_cost"],
      p_max_pu=data_wind['ESP'].iloc[:8760].values)

n.add("Generator", "solar", bus="Spain electricity", carrier="solar",
      p_nom_extendable=True, capital_cost=costs.at["solar", "capital_cost"],
      p_max_pu=data_solar['ESP'].iloc[:8760].values)

n.add("Load", "demand", bus="Spain electricity", 
      p_set=data_el['ESP'].iloc[:8760].values)

# Define the CO2 limit 
n.add("GlobalConstraint", "CO2Limit", 
      carrier_attribute="co2_emissions", 
      sense="<=", 
      constant=30e6)

print("Block 3: Components and CO2 Limit added.")

Block 3: Components and CO2 Limit added.


In [30]:
# Solve the model
n.optimize(solver_name='highs')

# Extract the shadow price (mu) 
required_co2_price = abs(n.global_constraints.at["CO2Limit", "mu"])

print(f"--- Task h Results ---")
print(f"Target CO2 Limit: {n.global_constraints.at['CO2Limit', 'constant']/1e6} MtCO2")
print(f"Required System-wide CO2 Price: {required_co2_price:.2f} EUR/tCO2")
print(f"Total System Cost: {n.objective / 1e9:.2f} Billion EUR")

C:\Users\20221122\AppData\Local\Temp\ipykernel_56084\2918316286.py:2: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize(solver_name='highs')
Index(['Spain electricity'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 2/2 [00:00<00:00, 304.19it/s]
INFO:linopy.io: Writing time: 0.06s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 35044 primals, 78847 duals
Objective: 1.60e+10
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper were not assigned to the network.


--- Task h Results ---
Target CO2 Limit: 30.0 MtCO2
Required System-wide CO2 Price: 12.12 EUR/tCO2
Total System Cost: 15.97 Billion EUR
